# 04 - Flower Runtime Orchestration

Rendered snapshot of how the deployment runtime is actuated in this demo. Captured at `2026-05-14T09:38:19` from the local Docker/Flower topology.

The goal of this notebook is not to retrain the model. It shows which SuperLinks, SuperNodes and SuperExec services are running, what each one connects to, and which `flwr run` calls drive the regional and global layers.

## 1. Running Services

There are three Flower federations running at the same time: `region-eu`, `region-na`, and `global`. Hospital SuperNodes connect to regional SuperLinks. Region gateway SuperNodes connect only to the global SuperLink.

In [1]:
!docker compose ps --format 'table {{.Service}}\t{{.State}}\t{{.Ports}}'

SERVICE                                   STATE     PORTS
flat-hospital-eu-01-superexec-clientapp   running   
flat-hospital-eu-01-supernode             running   
flat-hospital-eu-02-superexec-clientapp   running   
flat-hospital-eu-02-supernode             running   
flat-hospital-eu-03-superexec-clientapp   running   
flat-hospital-eu-03-supernode             running   
flat-hospital-na-01-superexec-clientapp   running   
flat-hospital-na-01-supernode             running   
flat-hospital-na-02-superexec-clientapp   running   
flat-hospital-na-02-supernode             running   
flat-hospital-na-03-superexec-clientapp   running   
flat-hospital-na-03-supernode             running   
flat-superexec-serverapp                  running   
flat-superlink                            running   0.0.0.0:49093->9093/tcp, [::]:49093->9093/tcp
global-superexec-serverapp                running   
global-superlink                          running   0.0.0.0:39093->9093/tcp, [::]:39093->9093/tcp
hosp

## 2. Compose Roles

`SuperLink` services expose the Flower APIs. `SuperExec ServerApp` services execute the server application submitted through `flwr run`. Hospital `SuperNode` services receive tasks from their regional SuperLink, while gateway SuperNodes receive tasks from the global SuperLink. ClientApp SuperExec services execute `hfl_cicids.client_app` next to each SuperNode.

In [2]:
from pathlib import Path
import pandas as pd
import yaml

def service_role(name):
    if name.endswith('-superlink'):
        return 'SuperLink'
    if name.endswith('-superexec-serverapp'):
        return 'SuperExec ServerApp'
    if name.endswith('-supernode'):
        return 'SuperNode'
    if name.endswith('-superexec-clientapp'):
        return 'SuperExec ClientApp'
    return 'service'

def list_arg(args, flag):
    return args[args.index(flag) + 1] if flag in args else ''

compose = yaml.safe_load(Path('docker-compose.yml').read_text())
rows = []
for name, service in sorted(compose['services'].items()):
    if name.startswith('flat-'):
        continue
    command = service.get('command', [])
    rows.append({
        'service': name,
        'role': service_role(name),
        'superlink_target': list_arg(command, '--superlink'),
        'appio_target': list_arg(command, '--appio-api-address'),
        'node_config': list_arg(command, '--node-config'),
        'volumes': ', '.join(service.get('volumes', [])),
    })
pd.DataFrame(rows)

                              service                role         superlink_target                     appio_target                                                                      node_config                                                     volumes
           global-superexec-serverapp SuperExec ServerApp                                     global-superlink:9091                                                                                                                             ./shared:/shared
                     global-superlink           SuperLink                                                                                                                                                                                                       
   hospital-eu-01-superexec-clientapp SuperExec ClientApp                             hospital-eu-01-supernode:9094                                                                                  ./data/partitions/hospital_eu_01

## 3. Flower Profiles

The profile names are what the orchestrator passes to `flwr run . <profile>`. Each profile points to one SuperLink control endpoint.

In [3]:
!flwr config list

Flower Config file: ~/.flwr/config.toml
SuperLink connections:
  region-eu (default)
  region-na
  global
  flat


## 4. Actuating the Federations

`scripts/run_hierarchical_rounds.py` is the control plane for the demo. For each global round it submits one regional Flower run per region, then one global Flower run where the clients are the gateway SuperNodes.

In [4]:
!python scripts/run_hierarchical_rounds.py --global-rounds 1 --regional-rounds 1 --batch-size 8192 --dry-run

──────────────────────────────── Global round 1 ────────────────────────────────
Running: flwr run . region-eu --stream --run-config 'level="regional" 
region="region_eu" global-round=1 
init-checkpoint="/shared/checkpoints/global/round_0.pt" 
output-checkpoint="/shared/checkpoints/region_eu/round_1.pt" num-server-rounds=1
local-epochs=1 batch-size=8192 learning-rate=0.001 input-dim=78 
region-num-examples=871173 fraction-train=1.0 fraction-evaluate=1.0 
min-train-nodes=3 min-evaluate-nodes=3 min-available-nodes=3'
Running: flwr run . region-na --stream --run-config 'level="regional" 
region="region_na" global-round=1 
init-checkpoint="/shared/checkpoints/global/round_0.pt" 
output-checkpoint="/shared/checkpoints/region_na/round_1.pt" num-server-rounds=1
local-epochs=1 batch-size=8192 learning-rate=0.001 input-dim=78 
region-num-examples=1110344 fraction-train=1.0 fraction-evaluate=1.0 
min-train-nodes=3 min-evaluate-nodes=3 min-available-nodes=3'
Running: flwr run . global --stream --

## 5. Runtime Logs

The hospital SuperNode receives train/evaluate messages from its regional SuperLink. The gateway SuperNode receives a train message from the global SuperLink and returns the regional checkpoint as a model update.

In [5]:
!docker compose logs --tail=12 hospital-eu-01-supernode region-eu-gateway-supernode region-eu-superlink global-superlink

$ docker compose logs --tail=8 hospital-eu-01-supernode
hospital-eu-01-supernode-1  | INFO :      
hospital-eu-01-supernode-1  | INFO :      [RUN 8892720515652009613]
hospital-eu-01-supernode-1  | INFO :      Receiving: evaluate message (ID: 226ada72ab4b62b4ba636a36a933269aa6e86eb78e93ebf4b6f6c11cea03790a)
hospital-eu-01-supernode-1  | INFO :      Received successfully
hospital-eu-01-supernode-1  | INFO :      
hospital-eu-01-supernode-1  | INFO :      [RUN 8892720515652009613]
hospital-eu-01-supernode-1  | INFO :      Sending: evaluate message (ID: c3bec7d8c8546c667c761a160828848adab0b974d2afc129faa9a54602a7eff5)
hospital-eu-01-supernode-1  | INFO :      Sent successfully

$ docker compose logs --tail=8 region-eu-gateway-supernode
region-eu-gateway-supernode-1  | INFO :      
region-eu-gateway-supernode-1  | INFO :      [RUN 7108604665246021897]
region-eu-gateway-supernode-1  | INFO :      Receiving: train message (ID: bcd8d96c4f1c22169205a0a50388bd6c01e202efbb4d1e61abe29d6cce9887b8)


## 6. Checkpoint Metadata

Regional checkpoints carry the regional training-example count used by global weighted FedAvg. The global checkpoint is produced by the global Flower federation.

In [6]:
from pathlib import Path
import json

for path in sorted(Path('shared/checkpoints').rglob('*.metadata.json')):
    print(path)
    print(json.dumps(json.loads(path.read_text()), indent=2, sort_keys=True))

shared/checkpoints/global/round_0.metadata.json
{
  "global_round": 0,
  "initialized": true,
  "input_dim": 78,
  "level": "global"
}
shared/checkpoints/global/round_1.metadata.json
{
  "global_round": 1,
  "input_dim": 78,
  "level": "global",
  "num_examples": 1981517,
  "weighted_by_key": "num-examples"
}
shared/checkpoints/region_eu/round_1.metadata.json
{
  "final_evaluate_metrics": {
    "eval_acc": 0.9088386543818298,
    "eval_auprc": 0.7388053754330514,
    "eval_balanced_acc": 0.9434234895798821,
    "eval_f1": 0.2398274571981538,
    "eval_false_negative_rate": 0.018916868113710777,
    "eval_false_positive_rate": 0.09423615272652494,
    "eval_precision": 0.15154821865302495,
    "eval_recall": 0.9810831318862892,
    "eval_roc_auc": 0.98831318865042
  },
  "final_train_metrics": {
    "train_loss": 0.44111061219332715,
    "val_auprc": 0.7370027252624184,
    "val_f1": 0.2748170929692166,
    "val_false_negative_rate": 0.019998102956170653,
    "val_false_positive_rate": 

## 7. Evaluation Snapshot

These metrics are from the latest rendered local run. They are evidence that the global checkpoint can be loaded and evaluated across the hospital test splits.

In [7]:
import pandas as pd
pd.read_csv('reports/metrics_summary_global.csv')

            run          model_kind          scope  weighted_f1  macro_f1  weighted_roc_auc  weighted_auprc  mean_false_positive_rate  worst_false_negative_rate  num_examples
hierarchical_fl hierarchical_global global_summary     0.668511  0.812493           0.98411        0.838231                  0.044571                   0.228034        424614


## 8. Privacy Boundary Check

`shared/` is allowed to contain checkpoints, metrics and preprocessing metadata. It should not contain hospital CSV/parquet rows.

In [8]:
from pathlib import Path
sorted(list(Path('shared').rglob('*.csv')) + list(Path('shared').rglob('*.parquet')))

[]
